# Fixed-crop preprocessing EDA

Explore raw BraTS geometry, justify one fixed 144-slice axial crop from measured evidence, and verify crop restoration without using project code.

## 1. Setup

Find every raw FLAIR and segmentation pair in HGG and LGG folders.

This cell builds `scans` only. Crop bounds stay undefined until the data is measured.

In [ ]:
# Import general tools only. Crop bounds are intentionally not defined here.
from pathlib import Path
from typing import NamedTuple

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from tqdm.auto import tqdm


# Discover canonical FLAIR and segmentation pairs directly from raw folders.
class Scan(NamedTuple):
    grade: str
    id: str
    flair: Path
    seg: Path


raw_dir = Path.home() / "datasets" / "brats_2019"
scans = []
for grade in ("HGG", "LGG"):
    for scan_dir in sorted((raw_dir / grade).iterdir()):
        if scan_dir.is_dir():
            scan_id = scan_dir.name
            scans.append(
                Scan(
                    grade,
                    scan_id,
                    scan_dir / f"{scan_id}_flair.nii",
                    scan_dir / f"{scan_id}_seg.nii",
                )
            )
print(f"Raw dataset: {raw_dir}")
print(f"Patients: {len(scans)}")

## 2. Raw dataset investigation

Measure raw shapes, spacing, brain bounds, and tumor bounds before choosing a crop.

This cell streams every patient, keeps compact measurements in `stats`, and reports dataset ranges and unusual masks.

In [ ]:
# Stream raw volumes and retain only compact geometry and per-slice counts.
class Stats(NamedTuple):
    grade: str
    id: str
    shape: tuple[int, ...]
    spacing: tuple[float, ...]
    brain_box: tuple[tuple[int, int], ...]
    tumor_box: tuple[tuple[int, int], ...]
    brain_z: np.ndarray
    tumor_z: np.ndarray
    outside: int


# Load one aligned pair directly from NIfTI files.
def read_pair(scan):
    flair = nib.as_closest_canonical(nib.load(scan.flair))
    seg = nib.as_closest_canonical(nib.load(scan.seg))
    assert flair.shape == seg.shape
    np.testing.assert_allclose(flair.affine, seg.affine, rtol=0.0, atol=1e-6)
    img = flair.get_fdata(dtype=np.float32, caching="unchanged")
    mask = np.asarray(seg.dataobj) > 0
    return img, mask, np.asarray(flair.affine)


def bbox(vol):
    coords = np.where(vol)
    return tuple((int(coord.min()), int(coord.max()) + 1) for coord in coords)


stats = []
for scan in tqdm(scans, desc="Investigating raw data", unit="patient"):
    img, mask, aff = read_pair(scan)
    brain = img != 0
    tumor = mask != 0
    stats.append(
        Stats(
            scan.grade,
            scan.id,
            tuple(img.shape),
            tuple(np.linalg.norm(aff[:3, :3], axis=0)),
            bbox(brain),
            bbox(tumor),
            brain.sum(axis=(0, 1)),
            tumor.sum(axis=(0, 1)),
            int((tumor & ~brain).sum()),
        )
    )


# Format depth statistics as one readable range.
def fmt_depth(vals):
    vals = np.asarray(vals)
    return (
        f"{vals.min():.0f}–{vals.max():.0f} slices"
        f" | median {np.median(vals):.1f}"
        f" | P05–P95 {np.percentile(vals, 5):.1f}–{np.percentile(vals, 95):.1f}"
    )


brain_depth = [stat.brain_box[2][1] - stat.brain_box[2][0] for stat in stats]
tumor_depth = [stat.tumor_box[2][1] - stat.tumor_box[2][0] for stat in stats]
shapes = sorted({stat.shape for stat in stats})
spacings = sorted({stat.spacing for stat in stats})
shape_txt = ", ".join(" × ".join(map(str, shape)) for shape in shapes)
spacing_txt = ", ".join(" × ".join(f"{val:.2f}" for val in spacing) for spacing in spacings)
outside_n = sum(stat.outside > 0 for stat in stats)

print("RAW DATASET SUMMARY")
print("Axial depth spans first through last nonzero Z slice for each patient.")
print()
print(f"{'Patients':<32} {len(stats)}")
print(f"{'Raw volume shape':<32} {shape_txt} voxels")
print(f"{'Raw voxel spacing':<32} {spacing_txt} mm")
print(f"{'FLAIR foreground Z depth':<32} {fmt_depth(brain_depth)}")
print(f"{'Tumor mask Z depth':<32} {fmt_depth(tumor_depth)}")
print(f"{'Mask beyond FLAIR foreground':<32} {outside_n} patients")
print("Mask mismatch means at least one tumor voxel lies where raw FLAIR equals zero.")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(brain_depth, bins=16)
ax[0].set(title="Brain axial depth", xlabel="Slices", ylabel="Patients")
ax[1].hist(tumor_depth, bins=16)
ax[1].set(title="Tumor axial depth", xlabel="Slices", ylabel="Patients")
plt.tight_layout()
plt.show()

## 3. Candidate crop analysis

Compare every possible contiguous 144-slice axial window.

Each row reports removed tumor and brain voxels, affected patients, and minimum tumor clearance.

In [ ]:
# Define compact result stored for each candidate window.
class CropStat(NamedTuple):
    start: int
    end: int
    tumor_lost: int
    tumor_cases: int
    brain_lost: int
    brain_cases: int
    clearance: int


# Compare every contiguous 144-slice window across all patients.
src_depth = stats[0].shape[2]
dst_depth = 144
crop_stats = []

for start in range(src_depth - dst_depth + 1):
    end = start + dst_depth
    brain_lost = [int(stat.brain_z[:start].sum() + stat.brain_z[end:].sum()) for stat in stats]
    tumor_lost = [int(stat.tumor_z[:start].sum() + stat.tumor_z[end:].sum()) for stat in stats]
    gaps = [min(stat.tumor_box[2][0] - start, end - stat.tumor_box[2][1]) for stat in stats]
    crop_stats.append(
        CropStat(
            start,
            end,
            sum(tumor_lost),
            sum(n > 0 for n in tumor_lost),
            sum(brain_lost),
            sum(n > 0 for n in brain_lost),
            min(gaps),
        )
    )

# Print comparable retention and clearance statistics.
print("Window   Tumor voxels  Tumor patients  Brain voxels  Brain patients  Min clearance")
for stat in crop_stats:
    print(
        f"{stat.start:>2}:{stat.end:<3}  {stat.tumor_lost:>12,}  "
        f"{stat.tumor_cases:>14}  {stat.brain_lost:>12,}  "
        f"{stat.brain_cases:>14}  {stat.clearance:>13}"
    )

## 4. Crop decision

Select fixed crop `5:149` from previous measurements.

The chosen window must contain 144 slices and remove no tumor voxels.

In [ ]:
# Record fixed crop only after raw investigation and candidate comparison.
SHAPE = (240, 240, 144)
Z_START = 5
Z_END = 149

choice = next(stat for stat in crop_stats if stat.start == Z_START and stat.end == Z_END)
assert Z_END - Z_START == SHAPE[2]
assert choice.tumor_lost == 0
print(f"Selected crop: [{Z_START}, {Z_END}) -> {SHAPE}")
print(choice)

## 5. Preprocessing check

Apply fixed crop and foreground z-score to patient whose tumor lies closest to crop boundary.

Checks cover shape, affine shift, binary labels, normalization, and unchanged source data.

In [ ]:
# Reproduce proposed preprocessing locally for exploration.
def zscore(img):
    fg = img != 0
    vals = img[fg]
    norm = np.zeros_like(img, dtype=np.float32)
    norm[fg] = (vals - vals.mean()) / max(vals.std(), 1e-8)
    return norm


def crop_pair(img, mask, aff):
    dst_img = zscore(img[:, :, Z_START:Z_END])
    dst_mask = mask[:, :, Z_START:Z_END].copy()
    dst_aff = aff.copy()
    dst_aff[:3, 3] += aff[:3, 2] * Z_START
    return dst_img, dst_mask, dst_aff


# Select patient whose tumor lies closest to either crop boundary.
sample_idx = min(
    range(len(stats)),
    key=lambda idx: min(
        stats[idx].tumor_box[2][0] - Z_START,
        Z_END - stats[idx].tumor_box[2][1],
    ),
)
sample = scans[sample_idx]
src_img, src_mask, src_aff = read_pair(sample)
src_before = src_img.copy()
dst_img, dst_mask, dst_aff = crop_pair(src_img, src_mask, src_aff)

# Check exact crop geometry, labels, normalization, and source immutability.
assert tuple(dst_img.shape) == tuple(dst_mask.shape) == SHAPE
np.testing.assert_allclose(dst_aff[:3, :3], src_aff[:3, :3])
np.testing.assert_allclose(dst_aff[:3, 3], src_aff[:3, 3] + src_aff[:3, 2] * Z_START)
np.testing.assert_array_equal(dst_mask, src_mask[:, :, Z_START:Z_END])
np.testing.assert_array_equal(src_img, src_before)
fg = dst_img != 0
np.testing.assert_allclose(dst_img[fg].mean(), 0.0, atol=1e-5)
np.testing.assert_allclose(dst_img[fg].std(), 1.0, atol=1e-5)
print(f"Preprocessing checks passed: {sample.grade}/{sample.id}")

## 6. Postprocessing check

Insert cropped mask back into original 155-slice frame.

Excluded slices remain zero, and perfect-mask round trip must return Dice `1.0`.

In [ ]:
# Restore one cropped prediction into its original source frame.
def restore(pred, src_shape):
    restored = np.zeros(src_shape, dtype=np.uint8)
    restored[:, :, Z_START:Z_END] = pred.astype(np.uint8)
    return restored


# Treat cropped target as a perfect prediction and restore source indices.
restored = restore(dst_mask, src_mask.shape)
expected = np.zeros(src_mask.shape, dtype=np.uint8)
expected[:, :, Z_START:Z_END] = dst_mask
np.testing.assert_array_equal(restored, expected)
assert restored[:, :, :Z_START].sum() == 0
assert restored[:, :, Z_END:].sum() == 0

truth = src_mask.astype(bool)
pred = restored.astype(bool)
dice = 2 * np.logical_and(truth, pred).sum() / (truth.sum() + pred.sum())
assert dice == 1.0
print(f"Postprocessing checks passed: shape={restored.shape}, Dice={dice:.3f}")

## 7. Whole-dataset verification

Run same preprocessing and postprocessing checks for every patient.

Summary reports lost foreground, affected patients, tumor clearance, and pipeline failures.

In [ ]:
# Verify complete preprocessing and postprocessing flow across all patients.
errors = []
for scan in tqdm(scans, desc="Verifying full pipeline", unit="patient"):
    src_img, src_mask, src_aff = read_pair(scan)
    dst_img, dst_mask, dst_aff = crop_pair(src_img, src_mask, src_aff)
    try:
        assert tuple(dst_img.shape) == tuple(dst_mask.shape) == SHAPE
        np.testing.assert_allclose(dst_aff[:3, :3], src_aff[:3, :3])
        np.testing.assert_array_equal(dst_mask, src_mask[:, :, Z_START:Z_END])
        restored = restore(dst_mask, src_mask.shape)
        np.testing.assert_array_equal(restored, src_mask)
    except (AssertionError, ValueError) as error:
        errors.append(f"{scan.grade}/{scan.id}: {error}")

# Summarize crop retention and full-pipeline results.
brain_total = sum(int(stat.brain_z.sum()) for stat in stats)
print(f"Patients verified: {len(scans)}")
print(f"Tumor voxels removed: {choice.tumor_lost:,}")
print(f"Patients losing tumor: {choice.tumor_cases}")
print(f"Brain voxels removed: {choice.brain_lost:,} ({choice.brain_lost / brain_total:.4%})")
print(f"Patients losing brain foreground: {choice.brain_cases}")
print(f"Minimum tumor boundary clearance: {choice.clearance} slices")
print(f"Pipeline failures: {len(errors)}")
assert not errors, errors[:5]

## 8. Visual inspection

Inspect one representative tumor slice at raw, cropped, and restored stages.

One three-panel row keeps visual comparison concise while showing exact alignment.

In [ ]:
# Select representative source slice with largest tumor area.
src_z = int(np.argmax(stats[sample_idx].tumor_z))
dst_z = src_z - Z_START

# Compare same anatomy before crop, after crop, and after restoration.
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
views = (
    (src_img[:, :, src_z], src_mask[:, :, src_z], f"Raw z={src_z}", "Greens"),
    (dst_img[:, :, dst_z], dst_mask[:, :, dst_z], f"Preprocessed z={dst_z}", "Greens"),
    (src_img[:, :, src_z], restored[:, :, src_z], f"Postprocessed z={src_z}", "Reds"),
)
for plot, (img, mask, title, color) in zip(ax, views, strict=True):
    plot.imshow(img, cmap="gray")
    plot.imshow(np.ma.masked_where(mask == 0, mask), cmap=color, alpha=0.45)
    plot.set_title(title)
    plot.axis("off")
fig.suptitle(f"{sample.grade}/{sample.id}")
plt.tight_layout()
plt.show()